# Batch Fermenter — Getting Started

This notebook walks through the canonical PyOMES pattern for a 0-D batch aerobic fermenter:

1. Configure via `FermenterBuilder` (five-line fluent API)
2. Attach a pH controller
3. Run the simulation and inspect the `BatchResult`
4. Plot time-series outputs

**Prerequisites:** `pip install -e .` from the repo root (exposes the `PyOMES` namespace).

## 1 · Environment setup

The cell below locates the repo root by searching upward for `pyproject.toml`,
then adds `models/` to `sys.path` so `vlmodels` is importable alongside the
editable `PyOMES` install.

In [ ]:
import sys
from pathlib import Path

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

_root = _find_root()
if str(_root / "models") not in sys.path:
    sys.path.insert(0, str(_root / "models"))

import numpy as np
import matplotlib.pyplot as plt
from vlmodels.fermenter.config import FermenterBuilder
from PyOMES.control.cv_loops import PHController

print("Imports OK - repo root:", _root)

## 2 · Parameters

These are the only values you need to change to explore different operating conditions.
Toggle `USE_PH_CONTROL` to compare buffered vs. unbuffered pH trajectories.

In [ ]:
USE_PH_CONTROL = True   # set False to see unbuffered pH drift
PH_SETPOINT    = 5.0    # PI controller target
TAU_H          = 5.0    # simulation duration (hours)
N_STEPS        = 1000   # number of output steps

## 3 · Build the simulation

`FermenterBuilder` composes a `ControlVolume` + `Simulation` from high-level
declarations. Each chained method configures one aspect:

| Method | What it sets |
|---|---|
| `.vessel(V_total_L, T_K)` | Reactor volume and temperature |
| `.gas_feed(vvm_min, composition)` | Sparge rate and inlet gas |
| `.transfer_kinetic(kLa_O2)` | Gas-liquid O2 transfer coefficient |
| `.chemistry()` | pH-active equilibria preset |
| `.organism(id)` | Biomass species |
| `.substrate(id, ...)` | Monod kinetics for growth on this substrate |
| `.controller(c)` | Attach a feedback controller |

In [ ]:
builder = (
    FermenterBuilder()
    .vessel(V_total_L=2000, T_K=305.15)
    .gas_feed(vvm_min=1.0, composition={"O2": 0.21, "N2": 0.79})
    .transfer_kinetic(kLa_O2=150.0)
    .chemistry()
    .organism("Yeast")
    .substrate("AceticAcid", mu_max=0.5, Ks=5e-3, yield_gX_gS=0.36)
    .label("batch_demo")
)

if USE_PH_CONTROL:
    builder = builder.controller(
        PHController(setpoint=PH_SETPOINT, Kp=0.5, Ki=0.0)
    )

print("Builder configured")

## 4 · Run

`build_simulation_and_run` is the one-shot path: it constructs the `Simulation` and
immediately calls `sim.run(tau_h, n_steps)`, returning a `BatchResult`.

In [ ]:
result = builder.build_simulation_and_run(tau_h=TAU_H, n_steps=N_STEPS)
print(f"Simulation finished in {result.runtime_s:.2f} s")

## 5 · Results summary

`BatchResult.liquid_mol[cv_key][species_id]` is a NumPy array of shape
`(n_steps + 1,)`. `BatchResult.pH[cv_key]` is the per-step pH array.

In [ ]:
cv = "main"

print(f"t = 0 to {TAU_H} h  |  {N_STEPS} steps  |  pH control = {USE_PH_CONTROL}")
print("
Final liquid inventory (mol):")
for sp in sorted(result.liquid_mol[cv]):
    print(f"  {sp:>18}: {result.liquid_mol[cv][sp][-1]:.4f}")

pH_final = result.pH[cv][-1]
print(f"
Final pH  : {pH_final:.3f}")
print(f"Runtime   : {result.runtime_s:.3f} s")

## 6 · Time-series plots

In [ ]:
cv  = "main"
t   = result.t_h
liq = result.liquid_mol[cv]
pH  = result.pH[cv]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f"Batch Fermenter  (pH control = {USE_PH_CONTROL})", fontsize=13)

ax = axes[0, 0]
ax.plot(t, liq.get("AceticAcid", np.zeros_like(t)), color="tab:orange")
ax.set(xlabel="Time (h)", ylabel="mol", title="Substrate (AceticAcid)")

ax = axes[0, 1]
ax.plot(t, liq.get("Yeast", np.zeros_like(t)), color="tab:green")
ax.set(xlabel="Time (h)", ylabel="mol", title="Biomass (Yeast)")

ax = axes[1, 0]
ax.plot(t, liq.get("O2", np.zeros_like(t)), color="tab:blue")
ax.set(xlabel="Time (h)", ylabel="mol", title="Dissolved Oxygen (liquid O2)")

ax = axes[1, 1]
ax.plot(t, pH, color="tab:red", label="pH")
if USE_PH_CONTROL:
    ax.axhline(PH_SETPOINT, ls="--", color="gray", label=f"setpoint ({PH_SETPOINT})")
ax.set(xlabel="Time (h)", ylabel="pH", title="pH")
ax.legend()

plt.tight_layout()
plt.show()